In [ ]:
from databricks.connect import DatabricksSession
spark = DatabricksSession.builder.getOrCreate()
from helper_utils.functions import add_initial_scd2_colummns, add_md5_hash_id
from delta.tables import DeltaTable
from yellow_taxi.yellow_taxi_utils import latest_lookup_snapshot, normalize_lookup, identify_scd2_changes, rows_to_close, rows_to_insert, run_scd2
from pyspark.sql.functions import col

In [ ]:
# variables
# interactive widgets for catalog and schema names

dbutils.widgets.text("catalog", "nyctaxi_project_dev")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("run_date", "2026-10-07") # default end date for testing

# dynamic variables for catalog and schema names

catalog = dbutils.widgets.get("catalog")
source_schema = dbutils.widgets.get("bronze_schema")
target_schema = dbutils.widgets.get("silver_schema")
run_date = dbutils.widgets.get("run_date")

source_table = f"{catalog}.{source_schema}.yellow_taxi_bronze_lookup"
target_table = f"{catalog}.{target_schema}.yellow_taxi_silver_lookup"

In [ ]:
# spark.sql(f"DROP TABLE {target_table}")

""


In [23]:
# load from bronze
bronze_lookup = spark.table(source_table)

# deduplicate and take only the last loaded record for each location

latest_lookup_load = latest_lookup_snapshot(bronze_lookup)

# cast data types
df_normalized = normalize_lookup(latest_lookup_load)


In [24]:
# initial load of data into target, to be skipped if table already exists 

if not spark.catalog.tableExists(target_table):
    df_initial = add_initial_scd2_colummns(df_normalized) # add scd2 metadata cols for the first load
    df_initial.write.mode("overwrite").saveAsTable(target_table)

In [29]:
# changed can occur to any of the 3 columns: borough, zone, service_zone

# scd2 columns
scd2_cols = ["borough", "zone", "service_zone"]
source_id_col = "s_hash"
target_id_col = "t_hash"

# create a hash for source df 
source_df_hash = add_md5_hash_id(df_normalized, source_id_col, scd2_cols)

# load silver lookup table
target_table_loaded = DeltaTable.forName(spark, target_table)
target_df = target_table_loaded.toDF()

# filter to exclude closed records
target_df_filtered = target_df.filter(col("is_current") == True)
target_df_hash = add_md5_hash_id(target_df_filtered, target_id_col, scd2_cols)

# identify changes 

# if source_hash != target_hash => change in the the upstream data, to be updated
# if target_hash is null => new row, to be inserted
# if source_hash is null => old row is missing, to be deleted

changes = identify_scd2_changes(source_df_hash, target_df_hash)



In [30]:
# identify rows to close (change_type = 'update', 'delete')

close_rows = rows_to_close(changes)
insert_rows = rows_to_insert(changes)

# stack together the changes
staged = close_rows.unionByName(insert_rows) 


In [ ]:
# close all records and insert new ones

run_scd2(target_table_loaded, staged, run_date)


In [ ]:
# create payment_type table to store mapping 

schema = "payment_type INT, description STRING"
payment_type_tbl = f"{catalog}.{target_schema}.payment_type"

payment_type = spark.createDataFrame(
    [(0, "Flex Fare trip"), (1, "Credit Card"), (2, "Cash"), (3, "No charge"), (4, "Dispute"), (5, "Unknown"), (6, "Voided trip")],
    schema=schema
)

# create table with payment types if not exists

if not spark.catalog.tableExists(payment_type_tbl):
    payment_type.write.mode("overwrite").saveAsTable(payment_type_tbl)

# spark.table("nyctaxi_project.silver.payment_type").show()

In [ ]:
# create rate table to store mapping 

schema = "rate_code_id INT, description STRING"
rate_tbl = f"{catalog}.{target_schema}.rates"

rates = spark.createDataFrame(
    [(1, "Standard rate"), (2, "JFK"), (3, "Newark"), (4, "Nassau or Westchester"), (5, "Negotiated fare"), (6, "Group ride trip"), (99, "Null/unknown")],
    schema=schema
)

# save as table
if not spark.catalog.tableExists(rate_tbl):
    rates.write.mode("overwrite").saveAsTable(rate_tbl)

# spark.table("nyctaxi_project.silver.rates").show()

In [ ]:
# create vendor table to store mapping 

schema = "vendor_id INT, description STRING"
vendor_tbl = f"{catalog}.{target_schema}.vendor"

vendor = spark.createDataFrame(
    [(1, "Creative Mobile Technologies, LLC"), (2, "Curb Mobility, LLC"), (6, "Myle Technologies Inc"), (7, "Helix"), (99, "Unknown")],
    schema=schema
)

# save as table
if not spark.catalog.tableExists(vendor_tbl):
    vendor.write.mode("overwrite").saveAsTable(vendor_tbl)

# spark.table("nyctaxi_project.silver.vendor").show()